In [ ]:
import os
import shutil
from typing import List, Optional
from pathlib import Path


def get_folder_size(folder_path: str) -> float:
    """计算文件夹大小（MB）"""
    total = 0
    for dirpath, dirnames, filenames in os.walk(folder_path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            if os.path.exists(fp):
                total += os.path.getsize(fp)
    return total / (1024 * 1024)  # MB


def _list_subdirs(path: str, prefix: Optional[str] = None) -> List[str]:
    """返回 path 下的子文件夹列表（按名称排序）。

    Parameters
    ----------
    prefix : str, optional
        若提供，只返回以该前缀开头的子文件夹（如 'ins'）。
    """
    items = os.listdir(path)
    result = []
    for f in items:
        if not os.path.isdir(os.path.join(path, f)):
            continue
        if f.startswith('.'):
            continue
        if prefix is not None and not f.startswith(prefix):
            continue
        result.append(f)
    return sorted(result)


In [ ]:
def list_scenario_contents(
    base_path: str,
    max_subfolders: int = 2,
    max_ins: int = 1
) -> None:
    """
    列出四层输出文件夹的结构和内容详情。

    结构: base_path / subfolder (r0_NE, ...) / ins* / scenario_folder (leuvenCRCollab...) / <files>

    Parameters
    ----------
    base_path : str
        输出文件夹根路径，如 'data/randomDemand20ReceiversOutput'
    max_subfolders : int
        显示详情的 subfolder (r*) 数量
    max_ins : int
        每个 subfolder 下显示详情的 ins* 数量
    """
    if not os.path.exists(base_path):
        print(f"路径不存在: {base_path}")
        return

    subfolders = _list_subdirs(base_path)
    total_size = get_folder_size(base_path)
    print(f"{'='*60}")
    print(f"基础路径: {base_path}")
    print(f"子文件夹总数 (r*): {len(subfolders)}")
    print(f"总大小: {total_size:.2f} MB ({total_size/1024:.2f} GB)")
    print(f"{'='*60}")

    for subfolder in subfolders[:max_subfolders]:
        subfolder_path = os.path.join(base_path, subfolder)
        ins_folders = _list_subdirs(subfolder_path, prefix='ins')
        subfolder_size = get_folder_size(subfolder_path)
        print(f"\n[subfolder] {subfolder}/ ({subfolder_size:.2f} MB, {len(ins_folders)} ins folders)")
        print("-" * 50)

        for ins in ins_folders[:max_ins]:
            ins_path = os.path.join(subfolder_path, ins)
            scenario_folders = _list_subdirs(ins_path)
            print(f"  [{ins}] ({len(scenario_folders)} scenario folders)")

            for scenario in scenario_folders:
                scenario_path = os.path.join(ins_path, scenario)
                scenario_size = get_folder_size(scenario_path)
                print(f"    <scenario> {scenario}/ ({scenario_size:.2f} MB)")

                contents = os.listdir(scenario_path)
                files = sorted(f for f in contents if os.path.isfile(os.path.join(scenario_path, f)))
                dirs = sorted(f for f in contents if os.path.isdir(os.path.join(scenario_path, f)))

                print(f"      文件 ({len(files)}):")
                for f in files[:15]:
                    fsize = os.path.getsize(os.path.join(scenario_path, f)) / 1024
                    print(f"        - {f} ({fsize:.1f} KB)")
                if len(files) > 15:
                    print(f"        ... 还有 {len(files)-15} 个文件")

                print(f"      文件夹 ({len(dirs)}):")
                for d in dirs:
                    dpath = os.path.join(scenario_path, d)
                    dsize = get_folder_size(dpath)
                    if d == 'ITERS':
                        iter_names = sorted(x for x in os.listdir(dpath) if x.startswith('it.'))
                        print(f"        - ITERS/ ({dsize:.2f} MB, {len(iter_names)} iterations: {iter_names[:5]}...)")
                    else:
                        print(f"        - {d}/ ({dsize:.2f} MB)")

        if len(ins_folders) > max_ins:
            print(f"  ... 还有 {len(ins_folders)-max_ins} 个 ins 文件夹未显示")


In [ ]:
def clean_output_folders(
    base_path: str,
    files_to_keep: List[str] = None,
    patterns_to_keep: List[str] = None,
    folders_to_keep: List[str] = None,
    iters_to_keep: List[int] = None,
    dry_run: bool = True,
    verbose: bool = True
) -> dict:
    """
    清理四层输出文件夹，只保留每个 scenario 文件夹内指定的文件和文件夹。

    结构: base_path / subfolder (r0_NE) / ins* / scenario_folder (leuvenCRCollab...) / <files>

    注意: scenario 文件夹本身（如 leuvenCRCollab...-i00）不会被删除，
    只清理其内部的文件 / 文件夹 / ITERS 迭代。

    Parameters
    ----------
    base_path : str
        输出文件夹根路径，如 'data/randomDemand20ReceiversOutput'
    files_to_keep : List[str], optional
        要保留的精确文件名列表
    patterns_to_keep : List[str], optional
        要保留的文件名模式（包含匹配）
    folders_to_keep : List[str], optional
        scenario 内要保留的文件夹名（除 ITERS 外）
    iters_to_keep : List[int], optional
        ITERS 内要保留的 iteration 编号，如 [0, 50]
    dry_run : bool
        True 时只预览，不实际删除
    verbose : bool
        是否显示每个文件的详情

    Returns
    -------
    dict : 统计信息
    """
    if files_to_keep is None:
        files_to_keep = [
            'output_carriers.xml.gz',
            'output_events.xml.gz',
            'carriers.xml.gz',
            'receivers.xml.gz',
            'receiver_stats.csv',
            'carrier_scores.txt',
            'receiver_scores.txt',
        ]

    if patterns_to_keep is None:
        patterns_to_keep = [
            'carrier_scores.png',
            'receiver_scores.png',
        ]

    if folders_to_keep is None:
        folders_to_keep = []

    if iters_to_keep is None:
        iters_to_keep = [0, 50]

    # ITERS 始终保留（内部会清理迭代）
    if 'ITERS' not in folders_to_keep:
        folders_to_keep = folders_to_keep + ['ITERS']

    iters_to_keep_names = {f'it.{i}' for i in iters_to_keep}
    files_to_keep_set = set(files_to_keep)

    def should_keep_file(filename: str) -> bool:
        if filename in files_to_keep_set:
            return True
        return any(p in filename for p in patterns_to_keep)

    def clean_scenario_folder(scenario_path: str) -> dict:
        """清理单个 scenario 文件夹内的内容（不删除 scenario 文件夹本身）。"""
        local = {'files_deleted': 0, 'folders_deleted': 0, 'bytes_freed': 0, 'files_kept': 0, 'errors': []}

        for item in os.listdir(scenario_path):
            item_path = os.path.join(scenario_path, item)

            if os.path.isfile(item_path):
                if should_keep_file(item):
                    local['files_kept'] += 1
                    if verbose:
                        print(f"      ✓ 保留: {item}")
                else:
                    fsize = os.path.getsize(item_path)
                    local['bytes_freed'] += fsize
                    local['files_deleted'] += 1
                    if verbose:
                        print(f"      ✗ 删除文件: {item} ({fsize/1024:.1f} KB)")
                    if not dry_run:
                        try:
                            os.remove(item_path)
                        except Exception as e:
                            local['errors'].append(f"删除失败 {item_path}: {e}")

            elif os.path.isdir(item_path):
                if item == 'ITERS':
                    for iter_folder in sorted(os.listdir(item_path)):
                        iter_path = os.path.join(item_path, iter_folder)
                        if not os.path.isdir(iter_path):
                            continue
                        if iter_folder in iters_to_keep_names:
                            if verbose:
                                print(f"      ✓ 保留 iteration: {iter_folder}")
                        else:
                            dsize = get_folder_size(iter_path) * 1024 * 1024
                            local['bytes_freed'] += dsize
                            local['folders_deleted'] += 1
                            if verbose:
                                print(f"      ✗ 删除 iteration: {iter_folder} ({dsize/1024/1024:.2f} MB)")
                            if not dry_run:
                                try:
                                    shutil.rmtree(iter_path)
                                except Exception as e:
                                    local['errors'].append(f"删除失败 {iter_path}: {e}")

                elif item in folders_to_keep:
                    if verbose:
                        print(f"      ✓ 保留文件夹: {item}/")
                else:
                    dsize = get_folder_size(item_path) * 1024 * 1024
                    local['bytes_freed'] += dsize
                    local['folders_deleted'] += 1
                    if verbose:
                        print(f"      ✗ 删除文件夹: {item}/ ({dsize/1024/1024:.2f} MB)")
                    if not dry_run:
                        try:
                            shutil.rmtree(item_path)
                        except Exception as e:
                            local['errors'].append(f"删除失败 {item_path}: {e}")

        return local

    # -------------------------------------------------------------------------
    stats = {
        'subfolders_processed': 0,
        'ins_processed': 0,
        'scenarios_processed': 0,
        'files_deleted': 0,
        'folders_deleted': 0,
        'bytes_freed': 0,
        'files_kept': 0,
        'errors': []
    }

    if not os.path.exists(base_path):
        print(f"路径不存在: {base_path}")
        return stats

    subfolders = _list_subdirs(base_path)
    mode_str = "[DRY RUN] " if dry_run else ""
    print(f"{mode_str}开始清理 {base_path}")
    print(f"保留文件:       {files_to_keep}")
    print(f"保留模式:       {patterns_to_keep}")
    print(f"保留文件夹:     {folders_to_keep}")
    print(f"保留iterations: {iters_to_keep}")
    print(f"subfolder 总数:  {len(subfolders)}")
    print("=" * 60)

    for subfolder in subfolders:
        subfolder_path = os.path.join(base_path, subfolder)
        ins_folders = _list_subdirs(subfolder_path, prefix='ins')
        stats['subfolders_processed'] += 1

        if verbose:
            print(f"\n[{subfolder}] ({len(ins_folders)} ins folders)")

        for ins in ins_folders:
            ins_path = os.path.join(subfolder_path, ins)
            scenario_folders = _list_subdirs(ins_path)
            stats['ins_processed'] += 1

            if verbose:
                print(f"  [{ins}] ({len(scenario_folders)} scenario folders)")

            for scenario in scenario_folders:
                scenario_path = os.path.join(ins_path, scenario)
                stats['scenarios_processed'] += 1

                if verbose:
                    print(f"    <scenario> {scenario}/")

                local = clean_scenario_folder(scenario_path)
                stats['files_deleted'] += local['files_deleted']
                stats['folders_deleted'] += local['folders_deleted']
                stats['bytes_freed'] += local['bytes_freed']
                stats['files_kept'] += local['files_kept']
                stats['errors'].extend(local['errors'])

    print("\n" + "=" * 60)
    print(f"{mode_str}清理统计:")
    print(f"  处理 subfolder 数: {stats['subfolders_processed']}")
    print(f"  处理 ins 文件夹数: {stats['ins_processed']}")
    print(f"  处理 scenario 数:  {stats['scenarios_processed']}")
    print(f"  保留文件数:        {stats['files_kept']}")
    print(f"  删除文件数:        {stats['files_deleted']}")
    print(f"  删除文件夹数:      {stats['folders_deleted']}")
    print(f"  释放空间:          {stats['bytes_freed']/1024/1024:.2f} MB ({stats['bytes_freed']/1024/1024/1024:.2f} GB)")

    if stats['errors']:
        print(f"\n错误 ({len(stats['errors'])}):")
        for err in stats['errors'][:10]:
            print(f"  - {err}")

    if dry_run:
        print(f"\n⚠️  这是 DRY RUN 模式，没有实际删除任何文件。")
        print(f"    设置 dry_run=False 来实际执行删除操作。")

    return stats


In [ ]:
# 配置路径
# 结构: base_path / r*_** / ins* / leuvenCRCollab...-iNN / <scenario files>
base_path = 'data/randomDemand20ReceiversOutput'

# 先查看文件夹结构
list_scenario_contents(base_path, max_subfolders=2, max_ins=1)


In [ ]:
# =============================================================================
# 配置保留策略
# =============================================================================

my_files_to_keep = [
    'output_carriers.xml.gz',
    'output_allVehicles.xml.gz',
    'output_config_reduced.xml',
    'output_events.xml.gz',
    'receivers.xml.gz',
    'receiver_stats.csv',
    'carrier_scores.txt',
    'receiver_scores.txt',
    'output_receiverInTourPlacement.csv.gz',
]

my_patterns_to_keep = [
    'carrier_scores.png',
    'receiver_scores.png',
]

my_folders_to_keep = [
    'analysis',
]  # ITERS 会自动加入

my_iters_to_keep = [0, 50]  # 保留的 iteration 编号


In [ ]:
# =============================================================================
# DRY RUN - 只预览，不删除
# =============================================================================
stats = clean_output_folders(
    base_path=base_path,
    files_to_keep=my_files_to_keep,
    patterns_to_keep=my_patterns_to_keep,
    folders_to_keep=my_folders_to_keep,
    iters_to_keep=my_iters_to_keep,
    dry_run=True,   # ⚠️ 设为 False 才会真正删除
    verbose=True    # 设为 False 只显示汇总
)


In [ ]:
# =============================================================================
# ⚠️ 实际删除 - 确认 DRY RUN 结果后再运行！
# =============================================================================
stats = clean_output_folders(
    base_path=base_path,
    files_to_keep=my_files_to_keep,
    patterns_to_keep=my_patterns_to_keep,
    folders_to_keep=my_folders_to_keep,
    iters_to_keep=my_iters_to_keep,
    dry_run=False,  # ⚠️ 这会真正删除文件！
    verbose=False
)


In [ ]:
# 删除后检查剩余结构
list_scenario_contents(base_path, max_subfolders=2, max_ins=1)
